# Variance & Standard Deviation — London Weather (Solution)

**Goal:** Use variance and standard deviation of temperature to decide the best month to visit London.

**Data:** ≈39 000 observations for London 2015 (`TemperatureC`, `month`, `hour`, …).

**Flowchart:**

![London Weather Variance Flowchart](london_weather_variance_flowchart.png)

---


## 0. Setup


In [ ]:
library(readr)
library(dplyr)
# library(ggplot2)  # optional

load("data/project.Rda")   # object: london_data

# Fallback:
# london_data <- read_csv("data/london_weather_2015.csv", show_col_types = FALSE)
# london_data$month <- sprintf("%02d", as.integer(london_data$month))

str(london_data)
cat("Rows:", nrow(london_data), "\n")


## 1. Explore the Data


In [ ]:
head(london_data)
names(london_data)
nrow(london_data)


## 2. Looking at Temperature (whole year)


In [ ]:
temp <- london_data$TemperatureC

average_temp <- mean(temp)
print(average_temp)

temperature_var <- var(temp)          # sample variance (N-1)
print(temperature_var)

# Population-style variance (optional)
temperature_var_pop <- mean((temp - mean(temp))^2)
print(temperature_var_pop)

temperature_standard_deviation <- sd(temp)
print(temperature_standard_deviation)

cat("\nInterpretation: the whole-year SD is large (~6.5 °C) mainly because of seasonal change.\n",
    "For trip planning we need month-specific SDs.\n")


## 3. Filtering by Month


In [ ]:
june <- london_data %>% filter(month == "06")
july <- london_data %>% filter(month == "07")

cat("June mean:", mean(june$TemperatureC), "  SD:", sd(june$TemperatureC), "\n")
cat("July mean:", mean(july$TemperatureC), "  SD:", sd(july$TemperatureC), "\n")

# June is slightly warmer; SDs are almost identical → both months are comparably stable.


In [ ]:
monthly_stats <- london_data %>%
  group_by(month) %>%
  summarise(mean = mean(TemperatureC),
            standard_deviation = sd(TemperatureC))

print(monthly_stats)

cat("\nCoolest month (lowest mean):", monthly_stats$month[which.min(monthly_stats$mean)], "\n")
cat("Most stable (lowest SD):", monthly_stats$month[which.min(monthly_stats$standard_deviation)], "\n")


### Visual summary


In [ ]:
# The following plots were pre-generated for the report:
# - london_temp_hist.png
# - london_monthly_mean_sd.png
# - london_weather_simulation.png

# Quick base-R view
par(mfrow = c(1, 2))
barplot(monthly_stats$mean, names.arg = monthly_stats$month,
        col = "coral", main = "Mean Temp by Month", ylab = "°C")
barplot(monthly_stats$standard_deviation, names.arg = monthly_stats$month,
        col = "skyblue", main = "SD of Temp by Month", ylab = "°C")


## 4. Alternate Code Paths


In [ ]:
# Alternate 1 — pure base R with tapply
means_base <- tapply(london_data$TemperatureC, london_data$month, mean)
sds_base   <- tapply(london_data$TemperatureC, london_data$month, sd)
monthly_base <- data.frame(month = names(means_base),
                           mean = as.numeric(means_base),
                           sd   = as.numeric(sds_base))
print(monthly_base)

# Alternate 2 — aggregate
agg_mean <- aggregate(TemperatureC ~ month, data = london_data, FUN = mean)
agg_sd   <- aggregate(TemperatureC ~ month, data = london_data, FUN = sd)
monthly_agg <- merge(agg_mean, agg_sd, by = "month", suffixes = c("_mean", "_sd"))
print(monthly_agg)

# Alternate 3 — split + lapply
by_month <- split(london_data$TemperatureC, london_data$month)
means_l  <- sapply(by_month, mean)
sds_l    <- sapply(by_month, sd)
print(data.frame(mean = means_l, sd = sds_l))


## 5. More Practice


In [ ]:
# 1. Rainiest month (proportion of rain observations)
rain_prop <- london_data %>%
  group_by(month) %>%
  summarise(rain_pct = mean(Precip == "rain") * 100) %>%
  arrange(desc(rain_pct))
print(rain_prop)

# 2. Humidity for coolest vs warmest month
cool_m <- monthly_stats$month[which.min(monthly_stats$mean)]
warm_m <- monthly_stats$month[which.max(monthly_stats$mean)]
hum_cool <- london_data %>% filter(month == cool_m) %>% pull(Humidity)
hum_warm <- london_data %>% filter(month == warm_m) %>% pull(Humidity)
cat("Coolest month", cool_m, " humidity mean/SD:", mean(hum_cool), sd(hum_cool), "\n")
cat("Warmest month", warm_m, " humidity mean/SD:", mean(hum_warm), sd(hum_warm), "\n")

# 3. Night (0-5) vs afternoon (12-17) temperature SD
night <- london_data %>% filter(hour %in% 0:5)
aft   <- london_data %>% filter(hour %in% 12:17)
cat("Night SD:", sd(night$TemperatureC), "  Afternoon SD:", sd(aft$TemperatureC), "\n")


## 6. Simulation Section

We vary two parameters:
1. Sample size (how many observations we draw).
2. Extra measurement noise (Gaussian SD added to temperature).


In [ ]:
set.seed(42)

# --- Sample-size simulation ---
sample_sizes <- c(100, 500, 2000, 10000, nrow(london_data))
sd_by_n <- sapply(sample_sizes, function(n) {
  sd(sample(london_data$TemperatureC, size = n))
})
print(data.frame(n = sample_sizes, sd = round(sd_by_n, 3)))

# --- Noise simulation ---
noise_levels <- seq(0, 4, by = 0.5)
sd_by_noise <- sapply(noise_levels, function(s) {
  sd(london_data$TemperatureC + rnorm(nrow(london_data), 0, s))
})
print(data.frame(noise_sd = noise_levels, observed_sd = round(sd_by_noise, 3)))

cat("\nObservation: larger samples give more stable SD estimates.\n",
    "Adding noise increases the observed SD roughly by the quadratic sum of SDs.\n")


In [ ]:
# Simple visualisation of the two simulations
par(mfrow = c(1, 2))
plot(sample_sizes, sd_by_n, type = "b", pch = 19,
     main = "SD vs sample size", xlab = "n", ylab = "SD (°C)")
plot(noise_levels, sd_by_noise, type = "b", pch = 19, col = "darkgreen",
     main = "SD after adding noise", xlab = "noise SD", ylab = "observed SD")


## 7. Audience-Aware Summary

**Analyst version**  
Whole-year mean temperature ≈ 10.3 °C, sample variance ≈ 41.6, SD ≈ 6.45 °C.  
Month-level SDs are tightly clustered around 3.3 °C; June mean 17.9 °C (SD 3.28), July mean 17.2 °C (SD 3.33).  
Base-R `tapply` / `aggregate` and dplyr `group_by` + `summarise` give identical results.  
Monte-Carlo shows SD estimates stabilise after a few thousand observations; additive noise inflates SD approximately as √(σ² + σ_noise²).

**Traveller / non-specialist version**  
London’s temperature changes a lot from winter to summer, so a single “average for the year” is not useful for packing.  
If you look month by month, June and July are both warm (around 17–18 °C) and equally steady — you are unlikely to get a sudden cold snap or heatwave that is much bigger than ±3 °C.  
Pick the month that matches the activities you want; the weather reliability is essentially the same.
